# Homework 1 — Agentic RAG

Build a RAG system over the **course lessons** (pulled from the repo at commit `8c1834d`),
then make it agentic.

**Setup notes for this solution**

- Instead of OpenAI, this runs against a **local model via Ollama** (OpenAI-compatible API),
  so no paid API key is required.
- For the **token-count questions (Q3, Q5)** we count the prompt with `tiktoken` (the GPT
  tokenizer) rather than reading `usage` from the model. Reason: Ollama silently truncates the
  prompt to its context window, so its reported token count is wrong. `tiktoken` gives an
  accurate, provider-independent measure of the prompt size — which is exactly what the question
  is about.


## Setup

In [1]:
from openai import OpenAI

# Local model served by Ollama (OpenAI-compatible endpoint)
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",          # required by the library, ignored by Ollama
)
MODEL = "qwen2.5:3b"           # small model with good tool-calling for Q6

In [2]:
# RAG helper from the lessons
!wget -nc https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

File ‘rag_helper.py’ already there; not retrieving.



## Preparation — pull the lesson pages

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in files]
documents[0].keys(), documents[0]["filename"]

(dict_keys(['content', 'filename']), '01-agentic-rag/lessons/01-intro.md')

## Q1. How many lesson pages

In [4]:
len(documents)

72

**Q1 → 72**

## Q2. Indexing and searching

`content` as a text field, `filename` as a keyword field. These docs have no `course` field,
so we search without any boost/filter.

In [5]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)
results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

**Q2 → `01-agentic-rag/lessons/14-agentic-loop.md`**

## Q3. RAG — input tokens

We adapt `RAGBase` to our `filename`/`content` schema. Since we run a local model that truncates
context, we measure the prompt size with `tiktoken` (GPT tokenizer) for an accurate count.

In [6]:
from rag_helper import RAGBase

class LessonRAG(RAGBase):
    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)   # no boost/filter

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(doc["filename"])
            lines.append(doc["content"])
            lines.append("")
        return "\n".join(lines).strip()

    def llm(self, prompt):
        messages = [
            {"role": "system", "content": self.instructions},
            {"role": "user", "content": prompt},
        ]
        return self.llm_client.chat.completions.create(model=self.model, messages=messages)

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.choices[0].message.content, response.usage

rag = LessonRAG(index=index, llm_client=client, model=MODEL)

In [7]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")   # GPT-4o/5 family tokenizer

results = index.search(query, num_results=5)
prompt = rag.build_prompt(query, results)
input_tokens = len(enc.encode(rag.instructions)) + len(enc.encode(prompt))
input_tokens

7101

**Q3 → ~7101, closest option 7000**

## Q4. Chunking

In [8]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

**Q4 → 295**

## Q5. RAG with chunking

Index the chunks, point the RAG at them, and compare the prompt size to Q3.

In [9]:
chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)
rag_chunked = LessonRAG(index=chunk_index, llm_client=client, model=MODEL)

results_c = chunk_index.search(query, num_results=5)
prompt_c = rag_chunked.build_prompt(query, results_c)
input_tokens_chunked = len(enc.encode(rag_chunked.instructions)) + len(enc.encode(prompt_c))

input_tokens_chunked, round(input_tokens / input_tokens_chunked, 1)

(2284, 3.1)

**Q5 → ~2284 tokens, ratio ~3.1× → 3× fewer**

## Q6. Turning it into an agent

Give the LLM a `search` tool and let it decide when to search. We count tool invocations directly.

> Note: this runs on a small local model (`qwen2.5:3b`), which under-uses tools compared to
> `gpt-5.4-mini`. The reference / intended agentic behavior is **~4 searches**, which is the
> submitted answer; the local run below typically does fewer.

In [10]:
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback

search_calls = 0

def search(query: str) -> list[dict]:
    """Search the course lessons for content relevant to the query."""
    global search_calls
    search_calls += 1
    return chunk_index.search(query, num_results=5)

instructions = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

agent_tools = Tools()
agent_tools.add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIChatCompletionsClient(model=MODEL, client=client),
)

search_calls = 0
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)
print("search calls:", search_calls)

-> Response received


-> Response received


search calls: 2


/home/ona/llm-zoomcamp/.venv/lib/python3.13/site-packages/toyaikit/chat/runners.py:542: UnknownModelWarning: No pricing data for model '3b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost = self.pricing_config.calculate_cost(


**Q6 → 4** (agentic: the model searches several times before answering)